In [0]:
%run ./config

In [0]:

def save_gold(df, table_name):
    (df.write.format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}"))

In [0]:
studies = spark.table(f"{CATALOG}.{SCHEMA}.silver_studies")

dim_study = (
    studies.select(F.sha2("nct_id", 256).alias("study_key"),
                   F.coalesce("phase", F.lit("NA")).alias("phase"),
                    "nct_id",
                    "study_title",
                    "study_type",
                    "overall_status",
                    "enrollment_type").dropDuplicates(["nct_id"]))

In [0]:
save_gold(dim_study, "gld_dim_study")

In [0]:
dim_sponsor = (
    studies.filter(F.col("sponsor_name").isNotNull())
    .select(F.sha2(F.concat_ws("|",F.lower(F.trim("sponsor_name")),F.coalesce(F.lower(F.trim("sponsor_class")),F.lit("unknown"))),256).alias("sponsor_key"),"sponsor_name","sponsor_class")
    .dropDuplicates(["sponsor_key"]))

save_gold(dim_sponsor, "dim_sponsor")

In [0]:
locations = spark.table(f"{CATALOG}.{SCHEMA}.silver_locations")
population = spark.table(f"{CATALOG}.{SCHEMA}.silver_population")

country_from_locations = (locations.filter(F.col("country_iso3").isNotNull()).select("country_iso3", "country_name"))

country_from_population = (population.select("country_iso3", "country_name"))

dim_country = (country_from_locations
    .unionByName(country_from_population)
    .filter(F.length("country_iso3") == 3)
    .withColumn("country_key",F.sha2("country_iso3", 256))
    .groupBy("country_key", "country_iso3")
    .agg(F.first("country_name",ignorenulls=True).alias("country_name")))

save_gold(dim_country, "dim_country")

In [0]:
interventions = spark.table(f"{CATALOG}.{SCHEMA}.silver_interventions")

dim_intervention = (interventions
    .select(F.sha2(F.concat_ws("|",F.lower(F.trim("intervention_type")),F.lower(F.trim("intervention_name"))),256).alias("intervention_key"),"intervention_type","intervention_name")
    .dropDuplicates(["intervention_key"]))

save_gold(dim_intervention, "dim_intervention")

In [0]:
date_limits = (studies.select(F.least(F.min("start_date"),F.min("completion_date")).alias("min_date"),
                              F.greatest(F.max("start_date"),F.max("completion_date")).alias("max_date")).first())

min_date = date_limits["min_date"]
max_date = date_limits["max_date"]

if min_date is None or max_date is None:
    raise ValueError("Não foi possível determinar o intervalo de datas.")

dim_date = (spark.sql(f"""SELECT EXPLODE(SEQUENCE(DATE('{min_date}'), DATE('{max_date}'),INTERVAL 1 DAY)) AS full_date""")
    .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),"full_date",
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.dayofmonth("full_date").alias("day"),
            F.weekofyear("full_date").alias("week_of_year")))

save_gold(dim_date, "dim_date")

In [0]:
fact_clinical_study = (
    studies
    .withColumn("study_key",F.sha2("nct_id", 256))
    .withColumn("sponsor_key",F.when(F.col("sponsor_name").isNotNull(),F.sha2(F.concat_ws("|",F.lower(F.trim("sponsor_name")),F.coalesce(F.lower(F.trim("sponsor_class")),F.lit("unknown"))),256)))
    .withColumn("start_date_key",F.date_format("start_date", "yyyyMMdd").cast("int"))
    .withColumn("completion_date_key",F.date_format("completion_date", "yyyyMMdd").cast("int"))
    .select("study_key",
            "sponsor_key",
            "start_date_key",
            "completion_date_key",
            "enrollment_count",
            "duration_days",
            "collected_at"))

save_gold(fact_clinical_study, "fact_clinical_study")

In [0]:
bridge_study_location = (
    locations
    .filter(F.col("country_iso3").isNotNull())
    .groupBy("nct_id", "country_iso3")
    .agg(F.countDistinct("facility_name","city").alias("facility_count"))
    .select(F.sha2("nct_id", 256).alias("study_key"),F.sha2("country_iso3", 256).alias("country_key"),"facility_count")
    .dropDuplicates(["study_key", "country_key"]))

save_gold(bridge_study_location,"bridge_study_location")

In [0]:
bridge_study_intervention = (
    interventions
    .select(F.sha2("nct_id", 256).alias("study_key"),F.sha2(F.concat_ws("|",F.lower(F.trim("intervention_type")),F.lower(F.trim("intervention_name"))),256).alias("intervention_key"))
    .dropDuplicates())

save_gold(bridge_study_intervention,"bridge_study_intervention")

In [0]:
fact_country_population = (
    population
    .select(F.sha2("country_iso3", 256).alias("country_key"),F.concat(F.col("year").cast("string"),F.lit("0101")).cast("int").alias("date_key"),"population")
    .dropDuplicates(["country_key", "date_key"]))

save_gold(fact_country_population,"fact_country_population")

In [0]:
country_year_studies = (
    bridge_study_location.alias("location")
    .join(
        fact_clinical_study.alias("fact"),
        "study_key"
    )
    .join(
        dim_country.alias("country"),
        "country_key"
    )
    .withColumn(
        "year",
        (F.col("fact.start_date_key") / 10000)
        .cast("int")
    )
    .groupBy(
        "country_key",
        "country.country_iso3",
        "country.country_name",
        "year"
    )
    .agg(
        F.countDistinct("study_key")
            .alias("study_count"),

        F.sum("facility_count")
            .alias("facility_count")
    )
)

country_year_metrics = (
    country_year_studies.alias("studies")
    .join(
        fact_country_population.alias("population"),
        (
            F.col("studies.country_key") ==
            F.col("population.country_key")
        ) &
        (
            F.col("population.date_key") ==
            F.concat(
                F.col("studies.year").cast("string"),
                F.lit("0101")
            ).cast("int")
        ),
        "left"
    )
    .select(
        "studies.country_key",
        "studies.country_iso3",
        "studies.country_name",
        "studies.year",
        "studies.study_count",
        "studies.facility_count",
        "population.population",

        F.round(
            F.col("studies.study_count") /
            F.col("population.population") *
            F.lit(1_000_000),
            4
        ).alias("studies_per_million")
    )
)

save_gold(
    country_year_metrics,
    "gold_country_year_metrics"
)

In [0]:
gold_tables = [
    "dim_study",
    "dim_sponsor",
    "dim_country",
    "dim_intervention",
    "dim_date",
    "fact_clinical_study",
    "bridge_study_location",
    "bridge_study_intervention",
    "fact_country_population",
    "gold_country_year_metrics"
]

for table_name in gold_tables:
    total = spark.table(
        f"{CATALOG}.{SCHEMA}.{table_name}"
    ).count()

    print(f"{table_name}: {total:,}")